In [2]:
import os

print(os.getcwd())

c:\Users\apran\OneDrive\Desktop\Olist_Customer_Experience_Analytics\notebooks\notebooks


In [3]:
import pandas as pd

orders = pd.read_csv("../../data/olist_orders_dataset.csv")
order_items = pd.read_csv("../../data/olist_order_items_dataset.csv")
reviews = pd.read_csv("../../data/olist_order_reviews_dataset.csv")

# Convert date columns
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

orders["order_delivered_customer_date"] = pd.to_datetime(
    orders["order_delivered_customer_date"]
)

orders["order_estimated_delivery_date"] = pd.to_datetime(
    orders["order_estimated_delivery_date"]
)

print("Data loaded successfully.")
print("Orders:", orders.shape)
print("Order items:", order_items.shape)
print("Reviews:", reviews.shape)

Data loaded successfully.
Orders: (99441, 8)
Order items: (112650, 7)
Reviews: (99224, 7)


In [4]:
# Delivery delay in days
orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 60 * 60)

# Late delivery flag
orders["late_delivery_flag"] = (
    orders["delivery_delay_days"] > 0
).astype(int)

print(orders[
    [
        "order_id",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_delay_days",
        "late_delivery_flag"
    ]
].head(10))

                           order_id order_delivered_customer_date  \
0  e481f51cbdc54678b7cc49136f2d6af7           2017-10-10 21:25:13   
1  53cdb2fc8bc7dce0b6741e2150273451           2018-08-07 15:27:45   
2  47770eb9100c2d0c44946d9cf07ec65d           2018-08-17 18:06:29   
3  949d5b44dbf5de918fe9c16f97b45f8a           2017-12-02 00:28:42   
4  ad21c59c0840e6cb83a9ceb5573f8159           2018-02-16 18:17:02   
5  a4591c265e18cb1dcee52889e2d8acc3           2017-07-26 10:57:55   
6  136cce7faa42fdb2cefd53fdc79a6098                           NaT   
7  6514b8ad8028c9f2cc2374ded245783f           2017-05-26 12:55:51   
8  76c6e866289321a7c93b82b54852dc33           2017-02-02 14:08:10   
9  e69bfb5eb88e0ed6a785585b27e16dbf           2017-08-16 17:14:30   

  order_estimated_delivery_date  delivery_delay_days  late_delivery_flag  
0                    2017-10-18            -7.107488                   0  
1                    2018-08-13            -5.355729                   0  
2              

In [5]:
freight_summary = (
    order_items.groupby("order_id")
    .agg(
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum")
    )
    .reset_index()
)

freight_summary["freight_ratio"] = (
    freight_summary["total_freight"]
    / freight_summary["total_price"]
)

print(freight_summary.head(10))

                           order_id  total_price  total_freight  freight_ratio
0  00010242fe8c5a6d1ba2dd792cb16214        58.90          13.29       0.225637
1  00018f77f2f0320c557190d7a144bdd3       239.90          19.93       0.083076
2  000229ec398224ef6ca0657da4fc703e       199.00          17.87       0.089799
3  00024acbcdf0a6daa1e931b038114c75        12.99          12.79       0.984604
4  00042b26cf59d7ce69dfabb4e55b4fd9       199.90          18.14       0.090745
5  00048cc3ae777c65dbb7d2a0634bc1ea        21.90          12.69       0.579452
6  00054e8431b9d7675808bcb819fb4a32        19.90          11.85       0.595477
7  000576fe39319847cbb9d288c5617fa6       810.00          70.75       0.087346
8  0005a1a1728c9d785b8e2b08b904576c       145.95          11.65       0.079822
9  0005f50442cb953dcd1d21e1fb923495        53.99          11.40       0.211150


In [6]:
orders = orders.merge(
    freight_summary[["order_id", "total_price", "total_freight", "freight_ratio"]],
    on="order_id",
    how="left"
)

print(orders[
    ["order_id", "total_price", "total_freight", "freight_ratio"]
].head(10))

                           order_id  total_price  total_freight  freight_ratio
0  e481f51cbdc54678b7cc49136f2d6af7        29.99           8.72       0.290764
1  53cdb2fc8bc7dce0b6741e2150273451       118.70          22.76       0.191744
2  47770eb9100c2d0c44946d9cf07ec65d       159.90          19.22       0.120200
3  949d5b44dbf5de918fe9c16f97b45f8a        45.00          27.20       0.604444
4  ad21c59c0840e6cb83a9ceb5573f8159        19.90           8.72       0.438191
5  a4591c265e18cb1dcee52889e2d8acc3       147.90          27.36       0.184990
6  136cce7faa42fdb2cefd53fdc79a6098        49.90          16.05       0.321643
7  6514b8ad8028c9f2cc2374ded245783f        59.99          15.17       0.252875
8  76c6e866289321a7c93b82b54852dc33        19.90          16.05       0.806533
9  e69bfb5eb88e0ed6a785585b27e16dbf       149.99          19.77       0.131809


In [7]:
# Merge review score into orders
review_summary = (
    reviews.groupby("order_id")
    .agg(review_score=("review_score", "mean"))
    .reset_index()
)

orders = orders.merge(
    review_summary,
    on="order_id",
    how="left"
)

# Normalize review score to 0–100
orders["review_score_normalized"] = (
    orders["review_score"] / 5
) * 100

# Delivery score: 100 for on-time/early, 0 for late
orders["delivery_score"] = (
    orders["late_delivery_flag"].apply(
        lambda x: 0 if x == 1 else 100
    )
)

# Customer Experience Score
orders["customer_experience_score"] = (
    0.7 * orders["review_score_normalized"] +
    0.3 * orders["delivery_score"]
)

print(
    orders[
        [
            "order_id",
            "review_score",
            "late_delivery_flag",
            "review_score_normalized",
            "delivery_score",
            "customer_experience_score"
        ]
    ].head(10)
)


                           order_id  review_score  late_delivery_flag  \
0  e481f51cbdc54678b7cc49136f2d6af7           4.0                   0   
1  53cdb2fc8bc7dce0b6741e2150273451           4.0                   0   
2  47770eb9100c2d0c44946d9cf07ec65d           5.0                   0   
3  949d5b44dbf5de918fe9c16f97b45f8a           5.0                   0   
4  ad21c59c0840e6cb83a9ceb5573f8159           5.0                   0   
5  a4591c265e18cb1dcee52889e2d8acc3           4.0                   0   
6  136cce7faa42fdb2cefd53fdc79a6098           2.0                   0   
7  6514b8ad8028c9f2cc2374ded245783f           5.0                   0   
8  76c6e866289321a7c93b82b54852dc33           1.0                   0   
9  e69bfb5eb88e0ed6a785585b27e16dbf           5.0                   0   

   review_score_normalized  delivery_score  customer_experience_score  
0                     80.0             100                       86.0  
1                     80.0             100          

In [8]:
# Add seller information to orders
seller_orders = order_items[
    ["order_id", "seller_id"]
].drop_duplicates()

seller_data = orders.merge(
    seller_orders,
    on="order_id",
    how="inner"
)

# Calculate seller metrics
seller_performance = (
    seller_data.groupby("seller_id")
    .agg(
        avg_review_score=("review_score", "mean"),
        on_time_rate=("late_delivery_flag", lambda x: (x == 0).mean() * 100),
        order_volume=("order_id", "nunique")
    )
    .reset_index()
)

# Normalize review score to 0–100
seller_performance["review_score_normalized"] = (
    seller_performance["avg_review_score"] / 5
) * 100

# Seller Performance Score
seller_performance["seller_performance_score"] = (
    0.5 * seller_performance["review_score_normalized"] +
    0.3 * seller_performance["on_time_rate"] +
    0.2 * (
        seller_performance["order_volume"] /
        seller_performance["order_volume"].max()
    ) * 100
)

print(seller_performance.head(10))

                          seller_id  avg_review_score  on_time_rate  \
0  0015a82c2db000af6aaaf3ae2ecb0532          3.666667    100.000000   
1  001cca7ae9ae17fb1caed9dfb1094831          3.984772     93.500000   
2  001e6ad469a905060d959994f1b41e4f          1.000000    100.000000   
3  002100f778ceb8431b7a1020ff7ab48f          3.901961     82.352941   
4  003554e2dce176b5555353e4f3555ac8          5.000000    100.000000   
5  004c9cd9d87a3c30c522c48c4fc07416          4.148387     91.772152   
6  00720abe85ba0859807595bbf045a33b          3.615385     84.615385   
7  00ab3eff1b5192e5f1a63bcecfee11c8          5.000000    100.000000   
8  00d8b143d12632bad99c0ad66ad52825          5.000000    100.000000   
9  00ee68308b45bc5e2660cd833c3f81cc          4.298507     91.111111   

   order_volume  review_score_normalized  seller_performance_score  
0             3                73.333333                 66.699029  
1           200                79.695431                 70.055213  
2          